# Skill translation: CV skills to English

Same step as `job_offer/2_translate_skills.ipynb`, for the CVs. `translate_skills_to_english`
(`pipeline/utils/mapping_utils.py`) sends each CV's skill list to `qwen3:8b` in one call, with the
CV's `job_category` as context, and returns one translation per skill.

- In: `mapping/preprocessed_cvs.json` (`id_c`, `job_category`, `skills`) from `0_preprocess_cvs.ipynb`.
- Out: `mapping/translated_skills.csv` (`id_c`, `raw_skills`, `eng_skills`).
- Next: `2_skill_extract.ipynb` maps `eng_skills` to the KB.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys; sys.path.append("../../../"); sys.path.append("../../utils")

import pandas as pd
from mapping_utils import pending, run_chunked, translate_skills_to_english

In [ ]:
OUT = ("mapping/translated_skills.csv", ["id_c", "raw_skills", "eng_skills"])

In [ ]:
preprocessed_cvs = pd.read_json("mapping/preprocessed_cvs.json")[["id_c", "job_category", "skills"]]
todo = pending(preprocessed_cvs, OUT)

In [ ]:
def translate_row(row):
    return {
        "id_c": row["id_c"],
        "raw_skills": row["skills"],
        "eng_skills": translate_skills_to_english(row["skills"], row["job_category"]),
    }

run_chunked(todo, translate_row, OUT)